# RAG Grid Search: 4-Config Experiment

Grid search across 2 axes: **chunk size** and **LLM model**.

Embedding (mpnet-base), retriever (dense/FAISS), reranker (`CrossEncoderReranker` ms-marco-MiniLM-L6, `top_n=5`), and search (`similarity`, `k=15`) held constant.

| Run | Chunk | Embedding | Retriever | LLM | Purpose |
|-----|-------|-----------|-----------|-----|---------|
| 1 | 256/32 | mpnet-base | dense | mistral-small (weak) | baseline |
| 2 | 512/64 | mpnet-base | dense | mistral-small (weak) | chunk size effect |
| 3 | 256/32 | mpnet-base | dense | llama-4-scout (strong) | LLM upgrade |
| 4 | 512/64 | mpnet-base | dense | llama-4-scout (strong) | dense+strong baseline |

In [1]:
import sys, os, json, re, random
from pathlib import Path
from typing import Dict, Any, List as listtype
from collections import defaultdict

from datasets import Dataset

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

from rapidfireai.automl import (
    List, RFLangChainRagSpec, RFOpenAIAPIModelConfig,
    RFPromptManager, RFGridSearch,
)

# Project roots (notebook runs under experiments/).
_repo_root = Path("..").resolve()
_discussion = _repo_root / "discussion"
_metrics = _repo_root / "Metrics"
if not _metrics.is_dir():
    raise FileNotFoundError(
        f"Expected Metrics/ at {_metrics} (cwd={Path.cwd()!s}). Fix cwd or symlink."
    )

# Datahub compat — must be called before importing Experiment
sys.path.insert(0, str(_repo_root))
sys.path.insert(0, str(_discussion))
sys.path.insert(0, str(_metrics))

# Ray actors run in worker processes — they inherit os.environ PYTHONPATH but not Jupyter's sys.path.
# Serialized metric callbacks import rapidfire_integration_example (Modules in Metrics/).
_md = str(_metrics)
_sep = os.pathsep
_pyp = os.environ.get("PYTHONPATH", "").strip()
if _md not in _pyp.split(_sep):
    os.environ["PYTHONPATH"] = f"{_md}{_sep}{_pyp}" if _pyp else _md

from rapidfireai_datahub_compat import patch_rapidfireai_for_datahub
print(patch_rapidfireai_for_datahub())

from rapidfireai import Experiment
from rapidfire_integration_example import (
    sample_compute_metrics_fn, sample_accumulate_metrics_fn,
)

TRITON_API_KEY = Path('~/api-key.txt').expanduser().read_text(encoding='utf-8').splitlines()[0].strip()
BASE_URL = 'https://tritonai-api.ucsd.edu'

os.environ["OPENAI_API_KEY"] = TRITON_API_KEY
os.environ["JUDGE_MODEL"] = "api-mistral-small-3.2-2506"
os.environ["JUDGE_BASE_URL"] = f"{BASE_URL}/v1"

INFO 04-27 18:47:42 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 04-27 18:47:42 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.
{'patched': True, 'dispatcher_proxy_url': '/hub/user-redirect/proxy/8851', 'is_jupyterhub_environment': True, 'dispatcher_host': '127.0.0.1', 'dispatcher_port': 8851, 'default_dispatcher_url': 'http://127.0.0.1:8851', 'recommended_proxy_url': '/hub/user-redirect/proxy/8851', 'jupyterhub_user': 'jovyan', 'jupyterhub_service_prefix': None}


TODO:

Check the need for instructions.

In [2]:
DATA_DIR = '../data/sourcedocs/'
VALIDATION_PATH = '../data/validation-set/validation-set-golden-qa-pairs.json'

batch_size = 32
DOWNSAMPLE_N = 30  # number of questions for experimentation; set to None for full set

WEAK_MODEL = 'api-mistral-small-3.2-2506'
STRONG_MODEL = 'api-llama-4-scout'

INSTRUCTIONS = """
You are a helpful assistant answering questions about RapidFire AI documentation.

Your job:
- Read the retrieved documentation context
- Answer the user's question using only the context
- Do not make up information that is not supported by the retrieved text

Rules:
1. Use only the retrieved document context.
2. Keep the answer concise but complete.
3. The answer should be in plain text and do not use any markdown syntax including list.
4. Mention relevant file names and the corresponding start and end line number when they are clearly useful.
5. There can be multiple sources but only keep the ones that are really relevant.
6. If there are sources from the same file, only keep the most relevant one.

Output format:
#### ANSWER: <final answer>
#### SOURCE: [{"file": <source file>, "lines": [<start line>, <end line>]}]
"""


class TextLoaderWithMetadata(TextLoader):
    def __init__(self, file_path, metadata_func=None, **kwargs):
        super().__init__(file_path=file_path, **kwargs)
        self.metadata_func = metadata_func

    def load(self):
        docs = super().load()
        if self.metadata_func is not None:
            for d in docs:
                d.metadata = self.metadata_func(d, d.metadata)
        return docs

    def lazy_load(self):
        for doc in super().lazy_load():
            if self.metadata_func is not None:
                doc.metadata = self.metadata_func(doc, dict(doc.metadata))
            yield doc


class TextSplitterWithLineNum(RecursiveCharacterTextSplitter):
    def split_documents(self, documents):
        split = super().split_documents(documents)
        line_num_map = {}
        for d in documents:
            file_name = d.metadata['source']
            m = defaultdict(list)
            for idx, line in enumerate(d.page_content.split('\n')):
                m[line.strip()].append(idx + 1)
            line_num_map[file_name] = m
        for chunk in split:
            source = chunk.metadata['source']
            lines = chunk.page_content.strip().split('\n')
            first = line_num_map[source][lines[0].strip()]
            last = line_num_map[source][lines[-1].strip()]
            start = first.pop(0)
            end = next(i for i in last if i >= start)
            chunk.metadata['source'] = {"file": source, "lines": [start, end]}
        return split


def metadata_func(document, metadata):
    metadata['source'] = Path(metadata['source']).name
    return metadata


def custom_template(doc):
    return f'{doc.metadata["source"]}: {doc.page_content}'

In [3]:
# --- Load corpus ---
loader = DirectoryLoader(
    path=DATA_DIR, glob='**/*.rst',
    loader_cls=TextLoaderWithMetadata,
    loader_kwargs={'metadata_func': metadata_func},
    show_progress=True, use_multithreading=False,
)
raw_docs = loader.load()
print(f"Loaded {len(raw_docs)} documents from {DATA_DIR}")

# --- Load validation set ---
with open(VALIDATION_PATH, 'r') as f:
    val_data = json.load(f)

if DOWNSAMPLE_N and DOWNSAMPLE_N < len(val_data):
    random.seed(42)
    val_data = random.sample(val_data, DOWNSAMPLE_N)
    print(f"Downsampled to {len(val_data)} questions for experimentation")

rows = []
for ex in val_data:
    rows.append({
        'question_id': ex['question_id'],
        'question': ex['question'],
        'reference_answer': ex['reference_answer'],
        'source_evidence': json.dumps(ex['source_evidence']),
    })

eval_dataset = Dataset.from_list(rows)
print(f"Eval dataset: {len(eval_dataset)} questions")
eval_dataset

100%|██████████| 31/31 [00:00<00:00, 2534.92it/s]

Loaded 31 documents from ../data/sourcedocs/
Downsampled to 30 questions for experimentation
Eval dataset: 30 questions


Dataset({
    features: ['question_id', 'question', 'reference_answer', 'source_evidence'],
    num_rows: 30
})

Splitter and embedding setup for the dense retrieval pipeline.

In [4]:
# --- Splitters ---
splitter_256 = TextSplitterWithLineNum.from_tiktoken_encoder(
    encoding_name='o200k_base', chunk_size=256, chunk_overlap=32,
    separators=['\n', '\n\n'],
)
splitter_512 = TextSplitterWithLineNum.from_tiktoken_encoder(
    encoding_name='o200k_base', chunk_size=512, chunk_overlap=64,
    separators=['\n', '\n\n'],
)

# --- Embedding config (dict form for RFLangChainRagSpec) ---
mpnet_embed_cfg = {
    'class': HuggingFaceEmbeddings,
    'model_name': 'sentence-transformers/all-mpnet-base-v2',
    'model_kwargs': {'device': 'cpu'},
    'encode_kwargs': {'normalize_embeddings': True, 'batch_size': batch_size},
}

print("Splitters and embedding config ready")

Splitters and embedding config ready


In [5]:
def grid_preprocess_fn(
    batch: Dict[str, listtype],
    rag: RFLangChainRagSpec,
    prompt_manager: RFPromptManager,
) -> Dict[str, listtype]:
    """Build prompts and extract retrieval metadata for metrics."""
    all_context = rag.get_context(batch_queries=batch['question'], serialize=False)

    # Extract (file, start, end) spans from each retrieved chunk
    retrieved_spans = []
    for docs in all_context:
        spans = []
        for doc in docs:
            src = doc.metadata.get('source')
            if isinstance(src, dict) and 'file' in src and 'lines' in src:
                spans.append((src['file'], src['lines'][0], src['lines'][1]))
        retrieved_spans.append(spans)

    serialized_context = rag.serialize_documents(all_context)

    # Parse ground-truth spans from the dataset
    ground_truth_spans = []
    for ev_str in batch.get('source_evidence', []):
        if isinstance(ev_str, str):
            ev_list = json.loads(ev_str)
        else:
            ev_list = ev_str
        spans = [(e['file'], e['lines'][0], e['lines'][1]) for e in ev_list]
        ground_truth_spans.append(spans)

    prompts = [
        [
            {'role': 'system', 'content': INSTRUCTIONS},
            {'role': 'user', 'content': f'Question: {q}\nDocumentation context: {ctx}\nYour response:'},
        ]
        for q, ctx in zip(batch['question'], serialized_context)
    ]

    return {
        'prompts': prompts,
        'query': batch['question'],
        'retrieved_spans': retrieved_spans,
        'serialized_context': serialized_context,
        'ground_truth_spans': ground_truth_spans,
        'reference_answer': batch.get('reference_answer', []),
        **batch,
    }


def grid_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    """Extract structured answer and source from raw LLM output."""
    answers, sources = [], []
    for text in batch['generated_text']:
        ans_match = re.search(
            r"####\s*Answer:\s*(.*?)(?=\n####\s*Source:|\Z)",
            text, flags=re.IGNORECASE | re.DOTALL,
        )
        src_match = re.search(
            r"####\s*Source:\s*(.*)",
            text, flags=re.IGNORECASE | re.DOTALL,
        )
        answers.append(ans_match.group(1).strip() if ans_match else text)
        sources.append(src_match.group(1).strip() if src_match else None)
    batch['answer'] = answers
    batch['source'] = sources
    return batch

In [6]:
# --- Shared config components ---
reranker_cfg = {
    'class': CrossEncoderReranker,
    'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2',
    'model_kwargs': {'device': 'cpu'},
    'top_n': 5,
}
vector_store_cfg = {'type': 'faiss', 'batch_size': batch_size}
search_cfg = {'type': 'similarity', 'k': 15}

client_cfg = {'api_key': TRITON_API_KEY, 'base_url': BASE_URL, 'max_retries': 2}

def make_openai_config(model_name, rag):
    return RFOpenAIAPIModelConfig(
        client_config=client_cfg,
        model_config={'model': model_name, 'max_completion_tokens': 2048, 'reasoning_effort': 'high'},
        rpm_limit=120,
        tpm_limit=1_000_000,
        rag=rag,
        prompt_manager=None,
    )

# ============================================================
# Dense RAG specs (configs 1-4): mpnet embedding, FAISS dense retriever
# ============================================================

# Config 1: chunk=256, mpnet, dense, WEAK
rag1 = RFLangChainRagSpec(
    document_loader=loader, text_splitter=splitter_256,
    embedding_cfg=mpnet_embed_cfg, vector_store_cfg=vector_store_cfg,
    search_cfg=search_cfg, reranker_cfg=reranker_cfg,
    enable_gpu_search=False, document_template=custom_template,
)

# Config 2: chunk=512, mpnet, dense, WEAK
rag2 = RFLangChainRagSpec(
    document_loader=loader, text_splitter=splitter_512,
    embedding_cfg=mpnet_embed_cfg, vector_store_cfg=vector_store_cfg,
    search_cfg=search_cfg, reranker_cfg=reranker_cfg,
    enable_gpu_search=False, document_template=custom_template,
)

# Config 3: chunk=256, mpnet, dense, STRONG
rag3 = RFLangChainRagSpec(
    document_loader=loader, text_splitter=splitter_256,
    embedding_cfg=mpnet_embed_cfg, vector_store_cfg=vector_store_cfg,
    search_cfg=search_cfg, reranker_cfg=reranker_cfg,
    enable_gpu_search=False, document_template=custom_template,
)

# Config 4: chunk=512, mpnet, dense, STRONG
rag4 = RFLangChainRagSpec(
    document_loader=loader, text_splitter=splitter_512,
    embedding_cfg=mpnet_embed_cfg, vector_store_cfg=vector_store_cfg,
    search_cfg=search_cfg, reranker_cfg=reranker_cfg,
    enable_gpu_search=False, document_template=custom_template,
)

# ============================================================
# OpenAI model configs (pairs RAG spec with LLM)
# ============================================================
oc1 = make_openai_config(WEAK_MODEL, rag1)
oc2 = make_openai_config(WEAK_MODEL, rag2)
oc3 = make_openai_config(STRONG_MODEL, rag3)
oc4 = make_openai_config(STRONG_MODEL, rag4)

CONFIG_LABELS = {
    1: "chunk=256, mpnet, dense, weak",
    2: "chunk=512, mpnet, dense, weak",
    3: "chunk=256, mpnet, dense, strong",
    4: "chunk=512, mpnet, dense, strong",
}

print(f"Built {len(CONFIG_LABELS)} RAG configs")

Built 4 RAG configs


In [7]:
# --- RFGridSearch config ---
config_set = [{
    'openai_config': List([oc1, oc2, oc3, oc4]),
    'batch_size': batch_size,
    'preprocess_fn': grid_preprocess_fn,
    'postprocess_fn': grid_postprocess_fn,
    'compute_metrics_fn': sample_compute_metrics_fn,
    'accumulate_metrics_fn': sample_accumulate_metrics_fn,
}]
config_group = RFGridSearch(config_set)

# --- Experiment ---
experiment = Experiment(
    experiment_name='rag-grid-search-4configs',
    mode='evals',
    experiment_path='../logs',
)
print("Experiment created. Ready to run.")

The previously running experiment rag-grid-search-4configs_2 was forcibly ended. Created a new experiment 'rag-grid-search-4configs_3' with Experiment ID: 13 at /home/emc001/rapidfireAI-rag-context-engineering/logs/rag-grid-search-4configs_3
Experiment created. Ready to run.


In [8]:
# --- Run the grid search ---
# This will launch 4 configs concurrently via RapidFire's adaptive execution.
results = experiment.run_evals(
    config_group=config_group,
    dataset=eval_dataset,
    num_shards=4,
    seed=42,
)

=== Preprocessing RAG Sources ===


RAG Source ID,Status,Duration,Device,Vector Store
1,Complete,127.7s,CPU,FAISS
2,Complete,127.7s,CPU,FAISS



=== Multi-Config Experiment Progress ===


Run ID,Model,Status,Progress,Conf. Interval,text_splitter,chunk_size,chunk_overlap,embedding_cfg.class,embedding_cfg.encode_kwargs.batch_size,embedding_cfg.encode_kwargs.normalize_embeddings,embedding_cfg.model_name,vector_store_cfg.batch_size,vector_store_cfg.type,search_cfg.k,search_cfg.type,reranker_cfg.class,reranker_cfg.model_name,reranker_cfg.top_n,model_config,Completeness (normalized),Correctness (pass rate),F1@5,Faithfulness (pass rate),Generation Score (3 released),Judge Failures,Precision@5,Processing Time,Recall@5,Retrieval Score,Samples Per Second,Samples Processed,Throughput,Total
1,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,TextSplitterWithLineNum,256,32,HuggingFaceEmbeddings,32,True,sentence-transformers/all-mpnet-base-v2,32,faiss,15,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=high","0.7467 [0.7467, 0.7467]","0.8333 [0.8333, 0.8333]","0.4198 [0.4198, 0.4198]","1.00 [1.00, 1.00]","0.8600 [0.8600, 0.8600]",0.0000,"0.2933 [0.2933, 0.2933]",384.82 seconds,"0.8250 [0.8250, 0.8250]","0.5127 [0.5127, 0.5127]",0.08,30,0.1/s,30
2,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,TextSplitterWithLineNum,512,64,HuggingFaceEmbeddings,32,True,sentence-transformers/all-mpnet-base-v2,32,faiss,15,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=high","0.7667 [0.7667, 0.7667]","0.9333 [0.9333, 0.9333]","0.3582 [0.3582, 0.3582]","1.00 [1.00, 1.00]","0.9000 [0.9000, 0.9000]",0.0000,"0.2333 [0.2333, 0.2333]",339.46 seconds,"0.8583 [0.8583, 0.8583]","0.4833 [0.4833, 0.4833]",0.09,30,0.1/s,30
3,api-llama-4-scout,COMPLETED,4/4,0.000,TextSplitterWithLineNum,256,32,HuggingFaceEmbeddings,32,True,sentence-transformers/all-mpnet-base-v2,32,faiss,15,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=high","0.7200 [0.7200, 0.7200]","0.9333 [0.9333, 0.9333]","0.4198 [0.4198, 0.4198]","1.00 [1.00, 1.00]","0.8844 [0.8844, 0.8844]",0.0000,"0.2933 [0.2933, 0.2933]",315.60 seconds,"0.8250 [0.8250, 0.8250]","0.5127 [0.5127, 0.5127]",0.10,30,0.1/s,30
4,api-llama-4-scout,COMPLETED,4/4,0.000,TextSplitterWithLineNum,512,64,HuggingFaceEmbeddings,32,True,sentence-transformers/all-mpnet-base-v2,32,faiss,15,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=high","0.7600 [0.7600, 0.7600]","0.9667 [0.9667, 0.9667]","0.3582 [0.3582, 0.3582]","1.00 [1.00, 1.00]","0.9089 [0.9089, 0.9089]",0.0000,"0.2333 [0.2333, 0.2333]",292.53 seconds,"0.8583 [0.8583, 0.8583]","0.4833 [0.4833, 0.4833]",0.10,30,0.1/s,30


In [9]:
# --- Track and export results ---
# Evals mode: pass eval_results from run_evals() — get_runs_info/get_results are fit-only.
sys.path.insert(0, str(Path('.').resolve()))
from track_results import dump_results

merged = dump_results(
    experiment,
    config_labels=CONFIG_LABELS,
    output_dir='../logs',
    eval_results=results,
)
merged

Results exported:
  ../logs/experiment_results.md
  ../logs/experiment_results.csv
  ../logs/experiment_results.json


,pipeline_id,label,run_id,model_name,text_splitter,chunk_size,chunk_overlap,embedding_cfg,vector_store_cfg,search_cfg,reranker_cfg,model_config,Samples Processed,Processing Time,Samples Per Second,Total,F1@5,Precision@5,Recall@5,Retrieval Score,Correctness (pass rate),Faithfulness (pass rate),Completeness (normalized),Generation Score (3 released),Judge Failures
0,1,"chunk=256, mpnet, dense, weak",1,api-mistral-small-3.2-2506,TextSplitterWithLineNum,256,32,"{""class"": ""HuggingFaceEmbeddings"", ""encode_kwa...","{""batch_size"": 32, ""type"": ""faiss""}","{""k"": 15, ""type"": ""similarity""}","{""class"": ""CrossEncoderReranker"", ""model_kwarg...","{""max_completion_tokens"": 2048, ""reasoning_eff...",30,384.82 seconds,0.08,30,"{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...",0
1,2,"chunk=512, mpnet, dense, weak",2,api-mistral-small-3.2-2506,TextSplitterWithLineNum,512,64,"{""class"": ""HuggingFaceEmbeddings"", ""encode_kwa...","{""batch_size"": 32, ""type"": ""faiss""}","{""k"": 15, ""type"": ""similarity""}","{""class"": ""CrossEncoderReranker"", ""model_kwarg...","{""max_completion_tokens"": 2048, ""reasoning_eff...",30,339.46 seconds,0.09,30,"{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...",0
2,3,"chunk=256, mpnet, dense, strong",3,api-llama-4-scout,TextSplitterWithLineNum,256,32,"{""class"": ""HuggingFaceEmbeddings"", ""encode_kwa...","{""batch_size"": 32, ""type"": ""faiss""}","{""k"": 15, ""type"": ""similarity""}","{""class"": ""CrossEncoderReranker"", ""model_kwarg...","{""max_completion_tokens"": 2048, ""reasoning_eff...",30,315.60 seconds,0.10,30,"{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...",0
3,4,"chunk=512, mpnet, dense, strong",4,api-llama-4-scout,TextSplitterWithLineNum,512,64,"{""class"": ""HuggingFaceEmbeddings"", ""encode_kwa...","{""batch_size"": 32, ""type"": ""faiss""}","{""k"": 15, ""type"": ""similarity""}","{""class"": ""CrossEncoderReranker"", ""model_kwarg...","{""max_completion_tokens"": 2048, ""reasoning_eff...",30,292.53 seconds,0.10,30,"{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...","{""ci_method"": ""normal"", ""confidence_interval"":...",0


In [10]:
# --- End experiment ---
experiment.end()
print("Experiment ended. Results saved to ../logs/")

Experiment rag-grid-search-4configs_3 ended
Experiment ended. Results saved to ../logs/
